# AutoIntel Pricing Pipeline Training

This notebook trains a deployable pricing pipeline for AutoIntel.

Design goal: the saved model accepts the same raw fields the API receives, then performs canonicalization, feature engineering, target encoding, one-hot encoding, and prediction inside the sklearn pipeline.

Output artifact: `../car_price_pipeline.pkl`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

START = Path.cwd().resolve()
API_ROOT = None
for candidate in [START, *START.parents]:
    if candidate.name == "api" and (candidate / "pyproject.toml").exists():
        API_ROOT = candidate
        break
    if (candidate / "apps" / "api" / "pyproject.toml").exists():
        API_ROOT = candidate / "apps" / "api"
        break
if API_ROOT is None:
    raise RuntimeError("Could not find apps/api from the current notebook directory.")
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

from app.ml_models.pricing_preprocessing import DEPLOY_INPUT_COLUMNS, build_pricing_pipeline

DATA_PATH = API_ROOT / "app" / "ml_models" / "data" / "car_prices_app_schema.csv"
MODEL_PATH = API_ROOT / "app" / "ml_models" / "car_price_pipeline.pkl"
REPORT_PATH = API_ROOT / "app" / "ml_models" / "pricing_pipeline_report.json"

DATA_PATH, MODEL_PATH

## Load App-Schema Training Data

The CSV already uses the same field names that the API sends to the pricing service.


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


## Training Filters

These filters remove unusable training rows and target outliers. They are intentionally outside the inference pipeline because inference must never drop a user's listing.


In [ ]:
before = len(df)
df = df.dropna(subset=["make", "model", "year", "price"]).copy()
df = df[(df["price"] >= 500) & (df["price"] <= 150_000)].copy()
df = df[(df["year"] >= 1980) & (df["year"] <= 2027)].copy()
df = df[(df["mileage"].isna()) | ((df["mileage"] >= 0) & (df["mileage"] <= 400_000))].copy()
df = df[(df["engine_volume"].isna()) | ((df["engine_volume"] > 0) & (df["engine_volume"] <= 8.5))].copy()
df = df[(df["engine_cylinders"].isna()) | (df["engine_cylinders"].isin([2, 3, 4, 5, 6, 8, 10, 12]))].copy()

print({"before": before, "after": len(df), "removed": before - len(df)})
df[DEPLOY_INPUT_COLUMNS + ["price"]].describe(include="all")

## Train/Test Split

In [ ]:
X = df[DEPLOY_INPUT_COLUMNS].copy()
y = df["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape, X_test.shape

## Train Pipeline

The pipeline owns all inference preprocessing. The API should pass app-shaped raw fields only.

In [ ]:
pipeline = build_pricing_pipeline(random_state=42, n_estimators=1000, n_jobs=-1)
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

metrics = {"mae": mae, "rmse": rmse, "r2": r2, "train_rows": len(X_train), "test_rows": len(X_test)}
metrics

## Smoke Test With App-Shaped Input

In [ ]:
sample = pd.DataFrame([{
    "make": "Mazda",
    "model": "CX-90",
    "year": 2024,
    "mileage": 25000,
    "body_type": "SUV",
    "transmission": "Automatic",
    "fuel_type": "Petrol",
    "drivetrain": "AWD",
    "engine_cylinders": 6,
    "engine_volume": 3.3,
    "color": "Black",
}], columns=DEPLOY_INPUT_COLUMNS)

int(round(float(pipeline.predict(sample)[0])))

## Save Artifact

Because custom transformers live in `app.ml_models.pricing_preprocessing`, the API can unpickle this artifact without notebook-local class hacks.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_PATH, compress=0)

report = {
    **metrics,
    "deploy_input_columns": DEPLOY_INPUT_COLUMNS,
    "model_path": str(MODEL_PATH),
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

MODEL_PATH, REPORT_PATH

## Reload Verification

In [ ]:
reloaded = joblib.load(MODEL_PATH, mmap_mode="r")
print(type(reloaded))
print(getattr(reloaded, "feature_names_in_", None))
print(int(round(float(reloaded.predict(sample)[0]))))